In [ ]:
import sys
import ast
import numpy as np
import pickle
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import pandas as pd
import os
import fnmatch



input_directories = ["/rdata/ian/pico/paperRuns/pinsga2_oneoffs/pinsga2_25to30_2000_2024-07-24_14-52-06", 
                    "/rdata/ian/pico/paperRuns/pinsga2_oneoffs/pinsga2_10to20_2000_2024-07-24_15-53-04",
                    "/rdata/ian/pico/paperRuns/pinsga2_oneoffs/pinsga2_20to30_2000_2024-07-24_16-07-04",
                    "/rdata/ian/pico/paperRuns/pinsga2_oneoffs/pinsga2_25to30_2000_2024-07-24_14-52-06"]


input_directories = [directory.rstrip('/') for directory in input_directories]

output_dir = "/rdata/ian/pico/paperRuns"
output_file = "%s/global_run_table.xlsx" % output_dir

raw_master_table = {'year':[], 'yield':[], 'irr_total':[], 'front':[], 'irrigation':[], 'run':[], 'gen':[], 'algorithm':[]}

In [ ]:
def parse_directory_name(full_path):

    file_name = full_path.split("/")[-1]
    
    fields = file_name.split("_")

    result = {}
    
    result["algorithm"] = fields[0]
    result["DM_range"] = fields[1]
    result["year"] = int(fields[2])
    result["run_date"] = fields[3]
    result["run_time"] = fields[4]

    return result

    

In [ ]:
def parse_file_name(file_name): 

    results = {}

    file_chunks = file_name.split("_")
    results["run"] = int(file_chunks[0][3:9])
    results["gen"] = int(file_chunks[1][3:9])

    return results 
    

In [ ]:
def get_file_names(directory, pattern): 

    matching_files = []
    for filename in os.listdir(directory):
        if fnmatch.fnmatch(filename, pattern):
            matching_files.append(os.path.join(directory, filename))
    
    return matching_files

In [ ]:
def read_files(files):

    all_solutions = None
    
    for (i, file_path) in enumerate(files):

        # Gather info from the file name
        file_name = file_path.split("/")[-1]
        full_dir = "/".join(file_path.split("/")[:-1])

        run_meta = parse_directory_name(full_dir)

        run_meta.update(parse_file_name(file_name) )
        # Get objective data
        current_objs = pd.read_csv(file_path, delimiter=',', names=["yield", "leaching"])

        # Pull in metadata on the run        
        current_objs["yield"]     = current_objs["yield"] * -1
        current_objs["run"]       = run_meta["run"]
        current_objs["gen"]       = run_meta["gen"]
        current_objs["algorithm"] = run_meta["algorithm"] 
        current_objs["DM_range"]  = run_meta["DM_range"]  
        current_objs["year"]      = run_meta["year"]      
        current_objs["run_date"]  = run_meta["run_date"]  
        current_objs["run_time"]  = run_meta["run_time"]  
                                    
        

        # Get decision variable data 
        var_file_path = file_path[:-7] + "var.csv"
        current_vars = pd.read_csv(var_file_path, delimiter=',', header=None)

        headers = ["var%s" % header for header in range(current_vars.shape[1])]
        current_vars = current_vars.set_axis(headers, axis=1)
        
        current_objs = pd.concat([current_objs,current_vars], axis=1)
        
        if all_solutions is None:
            all_solutions = current_objs
        else:
            all_solutions = pd.concat([all_solutions, current_objs])

    return all_solutions


In [ ]:
results = None

for directory in input_directories:


    dir_meta = parse_directory_name(directory)

    year = dir_meta["year"]
        
    print("Processing year %s for folder %s" % (year, directory))

    obj_files = get_file_names(directory, 'run*obj.csv')

    if results is None: 
        results = read_files(obj_files)
    else: 
        current_results = read_files(obj_files)
        result = pd.concat([results,current_results])
    
results.to_excel(output_file, engine='xlsxwriter')

print("Done")